In [1]:
# %% Settings

In [2]:
from pathlib import Path
import matplotlib

In [3]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [4]:
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

In [5]:
# %% Load data

In [6]:
df_alan = pd.read_excel("Göç etme nedenine göre illerin aldığı göç.xls", skiprows=4, header=None)

In [7]:
df_alan.columns = [
    "İl", "Toplam", "Tayin/iş değişikliği", "İş bulmak/işe başlamak",
    "Eğitim", "Evlilik", "Ailevi nedenler", "Daha iyi konut/yaşam",
    "Sağlık", "Hanedeki fertlere bağımlı", "Diğer", "Bilinmeyen",
    "Sütun12", "Sütun13"
]

In [8]:
df_veren = pd.read_excel("Göç etme nedenine göre illerin verdiği göç.xls", skiprows=4, header=None)
df_veren.columns = df_alan.columns

In [9]:
# Remove "Toplam-Total" row

In [10]:
df_alan = df_alan[df_alan["İl"] != "Toplam-Total"].copy()
df_veren = df_veren[df_veren["İl"] != "Toplam-Total"].copy()

In [11]:
# %% Build variables
panel = df_alan[["İl", "Toplam"]].rename(columns={"Toplam": "in_migration"})
panel["out_migration"] = df_veren["Toplam"].values
panel["net_migration"] = panel["in_migration"] - panel["out_migration"]
panel["net_migration_rate"] = panel["net_migration"] / panel["in_migration"] * 1000

In [12]:
# %% Quadrant analysis (2025 only)
panel["high_inmigration"] = panel["in_migration"] > panel["in_migration"].median()
panel["net_loss"] = panel["net_migration"] < 0

In [13]:
panel["quadrant"] = np.select(
    [panel["high_inmigration"] & panel["net_loss"],
     panel["high_inmigration"] & ~panel["net_loss"],
     ~panel["high_inmigration"] & panel["net_loss"]],
    ["High in-migration, net loss", "High in-migration, net gain",
     "Low in-migration, net loss"],
    default="Low in-migration, net gain")

In [14]:
quad = panel.groupby("quadrant").size()
print(quad)

quadrant
High in-migration, net gain    22
High in-migration, net loss    18
Low in-migration, net gain     12
Low in-migration, net loss     33
dtype: int64


In [15]:
# %% Figure: scatter of in-migration vs net migration
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(panel["in_migration"], panel["net_migration"], color="#264653", alpha=0.7)
ax.axhline(0, color="black", lw=0.8)
extreme = pd.concat([panel.nlargest(5, "in_migration"),
                     panel.nlargest(3, "net_migration"),
                     panel.nsmallest(3, "net_migration")]).drop_duplicates("İl")

In [16]:
for _, r in extreme.iterrows():
    ax.annotate(r["İl"], (r["in_migration"], r["net_migration"]),
                fontsize=8, xytext=(3, 3), textcoords="offset points")

In [17]:
ax.set_xlabel("In-migration (people)")
ax.set_ylabel("Net migration (people)")
ax.set_title("Turkish Provinces: In-Migration vs Net Migration (2025)")
plt.tight_layout()
plt.savefig(OUTPUT / "scatter_2025.png")
print("Figure saved!")

Figure saved!
